# Probability Bowl Model - World Cup 2026
Brier-score optimized. Market-first. Dixon-Coles corrected. Historical data backed.

### Pipeline
```
Historical rates (martj42) + Market odds + Lineups
  -> lambda estimation -> Dixon-Coles -> 300K Monte Carlo -> Props
```

### How to use
1. Run Cells 1-6 once per session (setup)
2. Fill in Cell 7 (match details) and run
3. Fill in Cell 8 (player props) and run

In [ ]:
# CELL 1: Install and imports
!pip install scipy numpy pandas requests --quiet

import numpy as np
import pandas as pd
import requests
import io
from scipy.stats import poisson
from scipy.optimize import minimize
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
N_SIMS = 300_000
print(f'Ready. Running {N_SIMS:,} simulations per match.')

In [ ]:
# CELL 2: Load historical data from martj42/international_results
print('Loading historical international results...')

RESULTS_URL = 'https://raw.githubusercontent.com/martj42/international_results/master/results.csv'

results_df = pd.read_csv(RESULTS_URL)
print(f'Loaded {len(results_df):,} matches')

# Filter to settled World Cup matches from 2010 onwards
wc = results_df[
    results_df['tournament'].str.contains('FIFA World Cup', na=False) &
    results_df['home_score'].notna() &
    (results_df['home_score'] != 'NA') &
    (results_df['date'] >= '2010-01-01')
].copy()

wc['home_score'] = pd.to_numeric(wc['home_score'], errors='coerce')
wc['away_score'] = pd.to_numeric(wc['away_score'], errors='coerce')
wc = wc.dropna(subset=['home_score', 'away_score'])

print(f'{len(wc)} settled World Cup matches from 2010+')
print(f'Avg goals/match: {(wc.home_score + wc.away_score).mean():.3f}')

In [ ]:
# CELL 3: Build team attack/defense profiles
home_rec = wc[['home_team', 'home_score', 'away_score']].rename(
    columns={'home_team': 'team', 'home_score': 'scored', 'away_score': 'conceded'})
away_rec = wc[['away_team', 'away_score', 'home_score']].rename(
    columns={'away_team': 'team', 'away_score': 'scored', 'home_score': 'conceded'})

all_records = pd.concat([home_rec, away_rec])
LEAGUE_AVG = all_records['scored'].mean()

team_stats = all_records.groupby('team').agg(
    avg_scored=('scored', 'mean'),
    avg_conceded=('conceded', 'mean'),
    n=('scored', 'count')
).reset_index()
team_stats['attack']  = team_stats['avg_scored']   / LEAGUE_AVG
team_stats['defense'] = team_stats['avg_conceded']  / LEAGUE_AVG

def get_profile(team):
    row = team_stats[team_stats['team'] == team]
    if len(row) == 0 or row.iloc[0]['n'] < 3:
        return {'attack': 1.0, 'defense': 1.0, 'n': 0}
    r = row.iloc[0]
    return {'attack': round(r['attack'],3), 'defense': round(r['defense'],3), 'n': int(r['n'])}

# Sanity check
for t in ['Brazil', 'Germany', 'Paraguay', 'Australia', 'Turkey', 'United States']:
    p = get_profile(t)
    print(f"{t:20s} | att={p['attack']:.3f} def={p['defense']:.3f} n={p['n']}")

print(f'\nLeague avg goals/team/match: {LEAGUE_AVG:.3f}')

In [ ]:
# CELL 4: Core model functions

def american_to_implied(odds):
    return 100/(odds+100) if odds > 0 else abs(odds)/(abs(odds)+100)

def remove_vig(home_ml, draw_ml, away_ml):
    raw = {k: american_to_implied(v)
           for k,v in zip(['home','draw','away'],[home_ml,draw_ml,away_ml])}
    total = sum(raw.values())
    fair = {k: v/total for k,v in raw.items()}
    print(f'  Vig: {(total-1)*100:.1f}% | '
          f'Home {fair["home"]:.1%} Draw {fair["draw"]:.1%} Away {fair["away"]:.1%}')
    return fair

def tau(x, y, lam, mu, rho):
    if   x==0 and y==0: return 1 - lam*mu*rho
    elif x==0 and y==1: return 1 + lam*rho
    elif x==1 and y==0: return 1 + mu*rho
    elif x==1 and y==1: return 1 - rho
    return 1.0

def dc_matrix(lam, mu, max_g=8, rho=-0.13):
    M = np.array([[tau(i,j,lam,mu,rho)*poisson.pmf(i,lam)*poisson.pmf(j,mu)
                   for j in range(max_g)] for i in range(max_g)])
    return M / M.sum()

def estimate_lambdas(home, away, p_home, p_draw, p_away,
                     avg_goals=2.4, market_weight=0.65, rho=-0.13):
    # Market-implied lambda
    def obj(params):
        lam, mu = params
        if lam<=0 or mu<=0: return 1e6
        M = dc_matrix(lam, mu, rho=rho)
        ph = np.tril(M,-1).sum()
        pd_ = np.diag(M).sum()
        pa = np.triu(M,1).sum()
        return (ph-p_home)**2 + (pd_-p_draw)**2 + (pa-p_away)**2

    ratio = (p_home+0.5*p_draw) / (p_away+0.5*p_draw+1e-6)
    lam0  = avg_goals*ratio/(1+ratio)
    mu0   = avg_goals - lam0
    res   = minimize(obj, [lam0,mu0], bounds=[(0.1,5),(0.1,5)], method='L-BFGS-B')
    lam_m, mu_m = res.x

    # Historical lambda
    hp = get_profile(home)
    ap = get_profile(away)
    lam_h = hp['attack'] * ap['defense'] * LEAGUE_AVG
    mu_h  = ap['attack'] * hp['defense'] * LEAGUE_AVG

    hw = (1-market_weight) if (hp['n']>=5 and ap['n']>=5) else 0.0
    mw = 1 - hw
    lam = mw*lam_m + hw*lam_h
    mu  = mw*mu_m  + hw*mu_h

    print(f'  Market lam: home={lam_m:.3f} away={mu_m:.3f}')
    if hw > 0:
        print(f'  History lam: home={lam_h:.3f} away={mu_h:.3f} (n={hp["n"]},{ap["n"]})')
    print(f'  Blended lam: home={lam:.3f} away={mu:.3f} (mkt={mw:.0%} hist={hw:.0%})')
    return lam, mu

print('Core model functions ready.')

In [ ]:
# CELL 5: Monte Carlo simulator and prop extractor

def simulate(lam, mu, cards_home=1.7, cards_away=1.7,
             corners_home=5.0, corners_away=5.0,
             offsides_home=1.8, offsides_away=1.8,
             shots_home=4.5, shots_away=4.5,
             rho=-0.13, n=N_SIMS):
    h1 = np.random.poisson(lam*0.45, n)
    a1 = np.random.poisson(mu*0.45,  n)
    h2 = np.random.poisson(lam*0.55, n)
    a2 = np.random.poisson(mu*0.55,  n)
    ht = h1+h2; at = a1+a2
    # DC correction on 0-0
    mask = (ht==0)&(at==0)
    reject = np.random.random(n) > (1-lam*mu*rho)
    rs = mask & reject
    ht[rs] = np.random.poisson(lam, rs.sum())
    at[rs] = np.random.poisson(mu,  rs.sum())
    return pd.DataFrame({
        'h1':ht-h2,'a1':at-a2,'h2':h2,'a2':a2,
        'ht':ht,'at':at,'tot':ht+at,
        'g1h':h1+a1,'g2h':h2+a2,
        'ch':np.random.poisson(cards_home,n),
        'ca':np.random.poisson(cards_away,n),
        'kh':np.random.poisson(corners_home,n),
        'ka':np.random.poisson(corners_away,n),
        'oh':np.random.poisson(offsides_home,n),
        'oa':np.random.poisson(offsides_away,n),
        'sh':np.random.poisson(shots_home,n),
        'sa':np.random.poisson(shots_away,n),
    })

def props(df, H='Home', A='Away'):
    btts = (df.ht>=1)&(df.at>=1)
    tc = df.ch+df.ca
    return {
        f'{H} win':              (df.ht>df.at).mean(),
        'Draw':                  (df.ht==df.at).mean(),
        f'{A} win':              (df.ht<df.at).mean(),
        f'{H} score 1+':         (df.ht>=1).mean(),
        f'{A} score 1+':         (df.at>=1).mean(),
        'BTTS':                  btts.mean(),
        'Over 1.5 goals':        (df.tot>=2).mean(),
        'Over 2.5 goals':        (df.tot>=3).mean(),
        'Under 2 goals':         (df.tot<=2).mean(),
        '3+ total goals':        (df.tot>=3).mean(),
        'BTTS + 3+ goals':       (btts&(df.tot>=3)).mean(),
        'HT draw':               (df.h1==df.a1).mean(),
        f'{H} leading HT':       (df.h1>df.a1).mean(),
        '2H more goals than 1H': (df.g2h>df.g1h).mean(),
        '2H has 2+ goals':       (df.g2h>=2).mean(),
        f'{H} score in 2H':      (df.h2>=1).mean(),
        f'{A} score in 2H':      (df.a2>=1).mean(),
        f'{H} more 2H goals':    (df.h2>df.a2).mean(),
        f'{A} more 2H goals':    (df.a2>df.h2).mean(),
        '3+ total cards':        (tc>=3).mean(),
        '4+ total cards':        (tc>=4).mean(),
        f'{H} more cards':       (df.ch>df.ca).mean(),
        f'{A} more cards':       (df.ca>df.ch).mean(),
        f'{H} 5+ corners':       (df.kh>=5).mean(),
        f'{A} 5+ corners':       (df.ka>=5).mean(),
        f'{H} more corners':     (df.kh>df.ka).mean(),
        f'{A} more corners':     (df.ka>df.kh).mean(),
        f'{H} offside 2+':       (df.oh>=2).mean(),
        f'{A} offside 2+':       (df.oa>=2).mean(),
        '8+ total SOT':          ((df.sh+df.sa)>=8).mean(),
    }

print('Simulator ready.')

In [ ]:
# CELL 6: Player SOT calculator + calibration layer

def player_sot(name, shots_per_90, sot_accuracy, expected_minutes,
               role_discount=1.0, context_factor=1.0):
    """
    P(player gets >= 1 SOT) via Poisson.

    role_discount cheat sheet:
      0.40  pure DM (Sangare, Xhaka holding, Tielemans)
      0.55  box-to-box double pivot (Kokcu today)
      0.70  wide midfielder / 8
      0.85  attacking mid / 10
      1.00  central striker / wide attacker
      1.15  penalty taker + set piece + sole creator (Enciso)

    context_factor:
      0.65  very cagey (both teams draw is enough)
      0.80  cautious
      1.00  neutral
      1.15  open game
      1.30  must-win mismatch
    """
    lam = shots_per_90 * sot_accuracy * (expected_minutes/90) * role_discount * context_factor
    prob = 1 - np.exp(-lam)
    print(f'  {name:25s} | lambda={lam:.3f} | P(>=1 SOT) = {prob:.1%}')
    return prob

# Calibration corrections from RBP history
CALIBRATION = {
    'penalty_red':       0.72,   # crowds say 35-40%, truth is 22-26%
    'btts_over25':       0.88,
    '2h_more_goals':     0.87,   # crowds overestimate 2H chaos
    'dm_sot':            0.45,   # deep midfielder SOT: biggest crowd overpricing edge
    'chasing_offsides':  1.15,   # chasing teams run behind more than model captures
    'big_fav_corners':   1.10,
    'underdog_score':    0.90,
}

def calibrate(raw, prop_type):
    c = float(np.clip(raw * CALIBRATION.get(prop_type, 1.0), 0.05, 0.95))
    print(f'  {prop_type}: {raw:.1%} -> {c:.1%} (x{CALIBRATION.get(prop_type,1.0)})')
    return c

print('Player SOT + calibration ready.')

In [ ]:
# CELL 7: FILL IN MATCH DETAILS HERE - edit for each match
# =====================================================
#  MATCH: Paraguay vs Australia | Jun 25 2026
# =====================================================

HOME = 'Paraguay'
AWAY = 'Australia'

# American moneyline odds (FanDuel / DraftKings)
HOME_ML = +165
DRAW_ML = +120
AWAY_ML = +175

# O/U line -> avg_goals proxy
# O/U 1.5 -> 1.8 | O/U 2.5 -> 2.4 | O/U 3.0 -> 2.9
AVG_GOALS = 1.8

# Referee yellows per game / 2 = per team rate
# Turpin: 3.39/game -> 1.7 per team
CARDS_HOME = 1.5
CARDS_AWAY = 1.5

# Corners: adjust for territorial dominance
# Paraguay avg only 0.5 corners/game in this WC
CORNERS_HOME = 2.5
CORNERS_AWAY = 4.5

# Offsides: higher for chasing teams
OFFSIDES_HOME = 1.4
OFFSIDES_AWAY = 1.9

# Shots on target rates
SHOTS_HOME = 3.5
SHOTS_AWAY = 4.5

# ---- RUN ----
print(f'\n{HOME} vs {AWAY}\n')

print('Historical WC profiles (2010+):')
for t in [HOME, AWAY]:
    p = get_profile(t)
    print(f'  {t}: attack={p["attack"]:.3f} defense={p["defense"]:.3f} n={p["n"]}')

print('\nMarket odds:')
fair = remove_vig(HOME_ML, DRAW_ML, AWAY_ML)

print('\nLambda estimation:')
lam, mu = estimate_lambdas(
    HOME, AWAY, fair['home'], fair['draw'], fair['away'],
    avg_goals=AVG_GOALS
)

print('\nRunning 300K simulations...')
df_sim = simulate(
    lam, mu,
    cards_home=CARDS_HOME, cards_away=CARDS_AWAY,
    corners_home=CORNERS_HOME, corners_away=CORNERS_AWAY,
    offsides_home=OFFSIDES_HOME, offsides_away=OFFSIDES_AWAY,
    shots_home=SHOTS_HOME, shots_away=SHOTS_AWAY,
)

p_all = props(df_sim, HOME, AWAY)

print('\nALL PROPS:')
out = pd.DataFrame([{'Prop': k, 'Probability': f'{v:.1%}'} for k, v in p_all.items()])
print(out.to_string(index=False))

In [ ]:
# CELL 8: PLAYER PROPS - fill in per match
print(f'Player props for {HOME} vs {AWAY}\n')

# player_sot(name, shots_per_90, sot_accuracy, minutes, role_discount, context)
# Add or remove players as needed

player_sot('Julio Enciso (PAR)',
    shots_per_90=2.9, sot_accuracy=0.36, expected_minutes=90,
    role_discount=1.15,   # penalty taker + set piece + sole creator
    context_factor=0.73   # cagey market but Paraguay must attack
)

# Deep midfielder example (Sangare/Xhaka/Tielemans pattern - always fade)
# player_sot('Deep DM example',
#     shots_per_90=1.8, sot_accuracy=0.35, expected_minutes=90,
#     role_discount=0.40,
#     context_factor=0.85
# )

print('\nCalibration check:')
calibrate(p_all.get('4+ total cards', 0.30), 'penalty_red')
calibrate(p_all.get('2H more goals than 1H', 0.34), '2h_more_goals')
calibrate(p_all.get(f'{AWAY} offside 2+', 0.44), 'chasing_offsides')

---
## Quick Reference

### Per-match checklist
```
1. Get moneyline odds from FanDuel/DraftKings
2. Check O/U line -> set AVG_GOALS
3. Look up referee on Transfermarkt -> set CARDS
4. Confirm lineups -> set role_discount for player props
5. Check suspensions/injuries -> adjust player props
6. Run Cell 7 -> all props
7. Run Cell 8 -> player SOT
8. Compare to crowd -> diverge only where justified
```

### Role discount cheat sheet
| Player type | role_discount |
|---|---|
| Pure DM (Sangare, Xhaka holding) | 0.40 |
| Box-to-box double pivot | 0.55 |
| Wide midfielder | 0.70 |
| Attacking mid / 10 | 0.85 |
| Central striker / winger | 1.00 |
| Penalty taker + set piece + sole creator | 1.15 |

### Calibration rules from RBP history
| Prop | Direction | Crowd | Your target |
|---|---|---|---|
| Penalty/red card | Fade | 35-42% | 22-26% |
| Deep midfielder SOT | Fade hard | 35-48% | 15-22% |
| Chasing team offsides | Boost | 35-45% | 48-55% |
| 2H more goals than 1H | Fade | 45-55% | 30-38% |
| BTTS + 3+ (competitive) | Floor | -- | min 28% |
